# samā4 — experiment setup

Loads the samā4 backend algorithms (ingestion pipeline + RAG context service) against the local
SQLite data in `minerva_dev.db`. Requires `OPENAI_API_KEY` and `PINECONE_API_KEY` in `.env`.

In [ ]:
import os
import sys
from pathlib import Path

# Run everything from the experiment root so `app` imports and relative data paths resolve.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
load_dotenv(ROOT.parent.parent / ".env")  # repo-root .env as fallback

# Default config targets Postgres; point at the bundled SQLite data instead.
os.environ.setdefault("DATABASE_URL", f"sqlite+aiosqlite:///{ROOT / 'minerva_dev.db'}")
os.environ.setdefault("ENVIRONMENT", "experiment")  # disables SQL echo

## Browse stored entities

In [ ]:
from sqlalchemy import select

from app.database import async_session
from app.models.entity import Entity

async with async_session() as db:
    entities = (await db.execute(select(Entity).limit(20))).scalars().all()

[(e.id, e.canonical_name) for e in entities]

## RAG context for one entity (cache → Pinecone → LLM)

In [ ]:
from app.services.context_service import ContextService

service = ContextService()
async with async_session() as db:
    context = await service.get_context(entities[0], db)

context

## Run the full ingestion pipeline (fetch → dedup → clean → NER → link → summarize → embed → store)

In [ ]:
from app.database import IS_SQLITE, create_tables
from app.ingestion.pipeline import IngestionPipeline

if IS_SQLITE:
    await create_tables()

count = await IngestionPipeline().run()
count